# Module 6: Deploy

> Self-directed module, ~15 min.

Ship the client research agent to **LangSmith Deployments** using the `langgraph` CLI. A deployed agent is reachable through 30+ endpoints — REST, MCP, A2A, Agent Protocol, HITL, persistent memory, and the Studio UI — without any additional plumbing.

**Steps:**

1. Inspect the deployable layout under `agents/deployable_agents/client_research/`
2. Review `langgraph.json` and the agent's `AGENTS.md`
3. Validate the configuration locally
4. Deploy to LangSmith


## Setup


In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(dotenv_path=project_root / ".env", override=True)

import os
print("LANGSMITH_API_KEY set:", bool(os.environ.get("LANGSMITH_API_KEY")))
print("OPENAI_API_KEY set:   ", bool(os.environ.get("OPENAI_API_KEY")))
print("TAVILY_API_KEY set:   ", bool(os.environ.get("TAVILY_API_KEY")))


## 1. Eval-safe vs deployable agents

The repo ships two parallel implementations of the client research agent:

| Path | Purpose | Notable differences |
|---|---|---|
| `agents/client_research_agent.py` | Eval-safe workhorse used by Modules 02–05 | No HITL, no FilesystemBackend, fresh `MemorySaver` per call |
| `agents/deployable_agents/client_research/` | The deployable variant (this module) | Adds `AGENTS.md`, two skills, `CompositeBackend(FilesystemBackend + StoreBackend)`, HITL on file writes |

Eval runs use the simpler version so file writes don't pause for human approval and disk state doesn't leak between runs. Production deployments use the richer variant so the agent has an editable identity, callable skills, persistent memory, and review gates on destructive operations.


## 2. Project structure

A deployable LangGraph project is a directory containing `langgraph.json` at the root that references one or more graph objects. `agents/deployable_agents/client_research/` holds the agent code, identity, and skills; `langgraph.json` (at the repo root) registers it.


In [ ]:
agent_dir = str(project_root / "agents" / "deployable_agents" / "client_research")
print("agents/deployable_agents/client_research/")
print("---")

for root, dirs, files in os.walk(agent_dir):
    dirs[:] = [d for d in dirs if d != "__pycache__"]
    level = root.replace(agent_dir, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")


## 3. Configuration

`langgraph.json` declares the deploy configuration: dependencies, graphs, and the env file. The LangGraph CLI reads it for both `langgraph dev` (local server) and `langgraph deploy` (push to LangSmith).

Two graphs are registered. `client_research` is the primary deployable for this module; `base_research_agent` is included as a second example for inspection.

`AGENTS.md` is the agent's identity — loaded into the prompt via the `memory=` argument in `create_deep_agent()` and editable by the agent itself at runtime.


In [ ]:
langgraph_json_path = project_root / "langgraph.json"
with open(langgraph_json_path) as f:
    print("langgraph.json:")
    print(f.read())

agents_path = os.path.join(agent_dir, "AGENTS.md")
with open(agents_path) as f:
    print("AGENTS.md:")
    print(f.read())


The agent itself lives at `agents/deployable_agents/client_research/agent.py`. The module-level `agent` variable is what gets deployed — `langgraph.json` references it as `"client_research": "./agents/deployable_agents/client_research/agent.py:agent"`.


## 4. Local development

Three CLI commands, all run from the repo root (where `langgraph.json` lives):

```bash
# Validate langgraph.json (imports each graph, checks dependencies)
langgraph validate

# Run locally with hot reload + Studio UI
langgraph dev --port 2024

# Deploy to LangSmith
langgraph deploy
```

`langgraph dev` opens the LangGraph Studio UI in the browser — useful for stepping through tool calls and approving HITL interrupts visually before deploying. By default it connects to `https://smith.langchain.com` for the Studio frontend and talks to the local server.

Reference: [LangGraph CLI](https://docs.langchain.com/langsmith/langgraph-cli) · [Deploy on Cloud](https://docs.langchain.com/langsmith/deploy-to-cloud).


## 5. Validate the configuration

`langgraph validate` imports each graph in `langgraph.json` and verifies dependencies resolve — without building Docker or uploading anything. Use it to catch configuration and import errors before deploying.


In [ ]:
# `cd` and `!` run in a subshell — chain in one line so cwd applies to the langgraph command.
!cd "{project_root}" && langgraph validate


## 6. Deploy to LangSmith

`langgraph deploy` builds a Docker image (locally if Docker is available, otherwise remotely on LangSmith's builder) and pushes it. Provisioning takes a few minutes.

> Requires a `LANGSMITH_API_KEY` with deployment permissions — a service key (`lsv2_sk_...`), not a personal token. On Apple Silicon, local builds need Docker Buildx; without Docker, the CLI falls back to a remote build automatically.

Useful flags:
- `--name <name>` — deployment name (defaults to the project directory name)
- `--deployment-type prod` — production deployment (default is `dev`)
- `--remote` — force remote build, skip local Docker
- `--no-wait` — return immediately rather than blocking on status


In [ ]:
# Re-run this command to push a new revision; the CLI finds the existing deployment by name.
# Add `--deployment-type prod` for production, or `--remote` to skip local Docker.
!cd "{project_root}" && langgraph deploy --name langsmith-poc-client-research --no-input


## 7. What you get with LangSmith Deployments

Once deployed, the agent is reachable through 30+ endpoints — built once, exposed everywhere:

| Capability | What it enables |
|---|---|
| **REST API** | Standard HTTP requests against `/runs`, `/threads`, `/store` |
| **Studio UI** | Visual debugger to step through state, threads, and tool calls |
| **Agent Protocol** | Stream runs and pause for human input |
| **MCP server** | Other agents can call this one as a tool |
| **A2A** | Agent-to-agent calls with handoffs |
| **Persistent Store** | `/memories/` survives restarts and threads via the platform's Store |
| **HITL** | Interrupt and resume from any client |
| **Cron / Scheduled runs** | Trigger the agent on a schedule |

Both registered graphs (`client_research` and `base_research_agent`) are reachable individually through these endpoints once the deployment is live.


## Recap

| File | Purpose |
|---|---|
| `langgraph.json` | LangGraph deploy config (dependencies, graphs, env file) |
| `agents/deployable_agents/client_research/agent.py` | The deployable `agent` object (built with `create_deep_agent`) |
| `AGENTS.md` | Agent identity and instructions (loaded via `memory=`) |
| `skills/` | On-demand capability modules — `client-brief`, `portfolio-update` (loaded via `skills=`) |
| `deepagents.toml` | Per-agent name and model config |
| `pyproject.toml` | Python dependencies for the deploy image |

**Operational follow-ups:**
- `langgraph deploy list` — list deployments
- `langgraph deploy logs <id>` — tail logs
- `langgraph deploy revisions list <id>` — revision history
- Re-run `langgraph deploy` to push a new revision; the CLI finds the existing deployment by name.

**Next:** continue iterating — Modules 02–05 cover the trace → online eval → annotation → experiment loop that runs against the deployed agent the same way it ran against the local one.
